# Graph-Based Network Intrusion Detection Using Graph Neural Networks
### Dataset: UNSW-NB15 | Mid-Semester Progress Demonstration

---

This notebook documents everything completed so far in our master's-level mini project.  
It uses **only actual saved results and project files** — no model retraining is performed here.

> ⚠️ **Important:** This system is **not** real-time packet capture.  
> The current pipeline accepts **CSV network-flow records** as input.

In [ ]:
# ── Setup: add project root to path so src/* imports work ──────────────────
import sys, os
from pathlib import Path

# Notebook lives in notebooks/; project root is one level up
PROJECT_ROOT = Path("__file__").resolve().parent.parent
# Works whether running from notebooks/ or from the project root directly
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "src" / "config.py").exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"src/config.py exists: {(PROJECT_ROOT / 'src' / 'config.py').exists()}")

In [ ]:
# ── Standard imports ────────────────────────────────────────────────────────
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Image, Markdown

matplotlib.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ── Convenience paths ───────────────────────────────────────────────────────
RESULTS_GNN  = PROJECT_ROOT / "results" / "gnn"
RESULTS_ML   = PROJECT_ROOT / "results" / "ml"
ABLATION_DIR = RESULTS_GNN / "ablation"
GRAPHS_DIR   = PROJECT_ROOT / "data" / "processed" / "graphs"
ANALYSIS_DIR = PROJECT_ROOT / "data" / "processed" / "graph_analysis"

print("All imports OK")

---
## 1. Dataset Loading & Inspection

The dataset is **UNSW-NB15** — a publicly available, labelled network-traffic benchmark  
created by the Australian Centre for Cyber Security.

Each row is one **network flow record** — a summary of a single network communication session.

In [ ]:
# ── Load raw CSVs (read-only) ────────────────────────────────────────────────
from src.config import TRAIN_RAW_PATH, TEST_RAW_PATH

train_raw = pd.read_csv(TRAIN_RAW_PATH, low_memory=False)
test_raw  = pd.read_csv(TEST_RAW_PATH,  low_memory=False)

print(f"Training set : {train_raw.shape[0]:>9,} rows × {train_raw.shape[1]} columns")
print(f"Testing set  : {test_raw.shape[0]:>9,} rows × {test_raw.shape[1]} columns")
print(f"Missing values (train) : {train_raw.isnull().sum().sum()}")
print(f"Duplicate rows (train) : {train_raw.duplicated().sum()}")

In [ ]:
# ── Column roles ─────────────────────────────────────────────────────────────
col_roles = pd.DataFrame([
    ("id",                    "Identifier",  "Row identifier — dropped before any ML step"),
    ("label",                 "Target",      "Binary: 0 = Normal, 1 = Attack"),
    ("attack_cat",            "Target",      "Multiclass: 10 attack categories"),
    ("proto",                 "Categorical", "Network protocol (tcp, udp, …)"),
    ("service",               "Categorical", "Application service (http, dns, …)"),
    ("state",                 "Categorical", "Connection state (FIN, CON, …)"),
    ("39 numerical columns",  "Numerical",   "Flow statistics: duration, bytes, packets, timing, ct_* counts…"),
], columns=["Column(s)", "Role", "Description"])

display(col_roles.style.set_properties(**{"text-align": "left"}).hide(axis="index"))

In [ ]:
# ── Label distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("UNSW-NB15 — Class Distributions", fontsize=13, fontweight="bold", y=1.02)

colors = ["#4CAF50", "#E53935"]

for ax, df, title in zip(axes,
                          [train_raw, test_raw],
                          ["Training Set (175,341 flows)", "Testing Set (82,332 flows)"]):
    vc = df["label"].value_counts().sort_index()
    bars = ax.bar(["Normal (0)", "Attack (1)"], vc.values, color=colors, width=0.5, edgecolor="white")
    for bar, val in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                f"{val:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("Flow count")
    ax.set_ylim(0, max(vc.values) * 1.12)

plt.tight_layout()
plt.show()

In [ ]:
# ── Attack category distribution (training set) ───────────────────────────────
cat_counts = train_raw["attack_cat"].value_counts()

fig, ax = plt.subplots(figsize=(11, 4))
palette = plt.cm.tab10(np.linspace(0, 1, len(cat_counts)))
bars = ax.bar(cat_counts.index, cat_counts.values, color=palette, edgecolor="white")
for bar, val in zip(bars, cat_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f"{val:,}", ha="center", va="bottom", fontsize=8)
ax.set_title("Attack Category Distribution — Training Set", fontsize=12, fontweight="bold")
ax.set_xlabel("Attack Category")
ax.set_ylabel("Count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

---
## 2. Data Preprocessing

Raw data cannot go directly into a neural network. We apply a **sklearn ColumnTransformer** pipeline:

| Feature type | Columns | Transformation |
|---|---|---|
| Numerical (39) | all numeric except id/targets | `StandardScaler` → mean=0, std=1 |
| Categorical (3) | `proto`, `service`, `state` | `OrdinalEncoder` → integer codes |
| Identifier (1) | `id` | **Dropped** — not a feature |
| Targets (2) | `label`, `attack_cat` | **Passed through unchanged** |

The fitted pipeline is saved to `data/processed/preprocessor.pkl` and is **never re-fitted** on test or inference data.

In [ ]:
# ── Load the fitted pipeline and apply to a small slice ──────────────────────
from src.preprocessor import load_pipeline, transform_split
from src.config import TARGET_COLS

pipeline = load_pipeline()
feature_names = list(pipeline.get_feature_names_out())

print(f"Pipeline output features : {len(feature_names)}")
print(f"Feature names (first 10) : {feature_names[:10]}")
print(f"\n→ 45 raw columns  ─ 1 id ─ 2 targets  =  42 model-input features")

In [ ]:
# ── Show a sample of the processed feature matrix (5 rows) ───────────────────
sample_raw  = train_raw.head(5).copy()
sample_proc = transform_split(pipeline, sample_raw, name="sample")
feat_cols   = [c for c in sample_proc.columns if c not in TARGET_COLS]

display(sample_proc[feat_cols[:8]].round(3)
        .style.set_caption("Processed feature matrix — first 5 rows, first 8 features")
        .format("{:.3f}"))

---
## 3. Graph Construction  (window = 100, k = 5)

### What is a graph in this project?

| Element | Definition |
|---|---|
| **Node** | One network-flow record (one CSV row) |
| **Node features** | 42 preprocessed values for that flow |
| **Edge** | Directed connection from a node to one of its k nearest neighbours |
| **Edge weight** | Cosine similarity score in the 8 `ct_*` feature dimensions |
| **Node label** | 0 = Normal, 1 = Attack |

> ⚠️ **This is NOT an IP-to-IP communication graph.**  
> The UNSW-NB15 CSV files used here do not contain `srcip`/`dstip` columns.  
> Edges represent **flow-similarity relationships**, not real network connections.

### Construction algorithm

```
Sort flows by id  →  divide into non-overlapping windows of 100 rows
For each window:
    For each node i:
        Compute cosine similarity to all other nodes using 8 ct_* features
        Connect i → its 5 most similar neighbours  (k=5)
        Store similarity score as edge weight
    Save as PyG Data object (.pt file)
```

### The 8 ct_* edge features
These count how many similar connections occurred in the last 100 flows,  
making them the closest available proxy for "relatedness" between flows.

In [ ]:
# ── Display the 8 ct_* features used for edge construction ───────────────────
from src.config import CT_EDGE_FEATURES

ct_df = pd.DataFrame([
    ("ct_srv_src",       "Connections from same source to same service (last 100)"),
    ("ct_srv_dst",       "Connections to same destination with same service (last 100)"),
    ("ct_src_ltm",       "Connections from same source (last 100)"),
    ("ct_dst_ltm",       "Connections to same destination (last 100)"),
    ("ct_dst_src_ltm",   "Connections between same source-destination pair (last 100)"),
    ("ct_src_dport_ltm", "Connections from same source to same destination port (last 100)"),
    ("ct_dst_sport_ltm", "Connections to same destination from same source port (last 100)"),
    ("ct_state_ttl",     "Connections with same protocol, state, TTL (last 100)"),
], columns=["Feature", "Meaning"])

display(ct_df.style.hide(axis="index"))

---
## 4. Graph Statistics & Validation

Statistics loaded from `data/processed/graphs/overall_stats.json` (generated during Phase 2).

In [ ]:
# ── Load overall graph statistics ────────────────────────────────────────────
with open(GRAPHS_DIR / "overall_stats.json") as f:
    overall = json.load(f)

tr, te = overall["train"], overall["test"]

stats_df = pd.DataFrame([
    ["Graphs",              f"{tr['n_graphs']:,}",    f"{te['n_graphs']:,}"],
    ["Total nodes",         f"{tr['total_nodes']:,}", f"{te['total_nodes']:,}"],
    ["Total edges",         f"{tr['total_nodes']*5:,}",  f"{te['total_nodes']*5:,}"],
    ["Attack nodes",        f"{tr['total_attack_nodes']:,} ({tr['attack_node_pct']:.1f}%)",
                            f"{te['total_attack_nodes']:,} ({te['attack_node_pct']:.1f}%)"],
    ["Normal nodes",        f"{tr['total_normal_nodes']:,}", f"{te['total_normal_nodes']:,}"],
    ["Avg edges/graph",     f"{tr['n_edges_mean']:.1f}",  f"{te['n_edges_mean']:.1f}"],
    ["Avg degree/node",     f"{tr['avg_degree_mean']:.1f}", f"{te['avg_degree_mean']:.1f}"],
    ["Isolated nodes",      f"{tr['n_isolated_mean']:.0f}", f"{te['n_isolated_mean']:.0f}"],
    ["Avg edge weight",     f"{tr['ew_mean_mean']:.4f}", f"{te['ew_mean_mean']:.4f}"],
], columns=["Statistic", "Training", "Testing"])

display(stats_df.style.hide(axis="index"))

In [ ]:
# ── Visualise one real graph (graph_00001.pt — mix of Normal+Attack nodes) ────
import torch
try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False

# Load a graph that has both Normal and Attack nodes
graph_path = GRAPHS_DIR / "train" / "graph_00001.pt"
g = torch.load(graph_path, weights_only=False)

print(f"Loaded graph_00001.pt")
print(f"  Nodes          : {g.num_nodes}")
print(f"  Edges          : {g.edge_index.shape[1]}")
print(f"  Node feature dim: {g.x.shape[1]}")
print(f"  Normal nodes   : {(g.y == 0).sum().item()}")
print(f"  Attack nodes   : {(g.y == 1).sum().item()}")
print(f"  Edge weight range: [{g.edge_attr.min():.3f}, {g.edge_attr.max():.3f}]")

In [ ]:
# ── Draw a small subgraph (first 30 nodes for clarity) ───────────────────────
if HAS_NX:
    N_SHOW = 30
    ei = g.edge_index.numpy()
    labels_np = g.y.numpy()

    # Keep only edges where both endpoints are within the first N_SHOW nodes
    mask = (ei[0] < N_SHOW) & (ei[1] < N_SHOW)
    sub_src = ei[0][mask].tolist()
    sub_dst = ei[1][mask].tolist()

    G_nx = nx.DiGraph()
    G_nx.add_nodes_from(range(N_SHOW))
    G_nx.add_edges_from(zip(sub_src, sub_dst))

    node_colors = ["#E53935" if labels_np[i] == 1 else "#43A047" for i in range(N_SHOW)]

    fig, ax = plt.subplots(figsize=(10, 7))
    pos = nx.spring_layout(G_nx, seed=42, k=1.2)
    nx.draw_networkx(
        G_nx, pos=pos, ax=ax,
        node_color=node_colors, node_size=220,
        edge_color="#90A4AE", arrows=True,
        arrowsize=8, width=0.7,
        with_labels=True, font_size=7, font_color="white",
    )
    legend_handles = [
        mpatches.Patch(color="#43A047", label="Normal (0)"),
        mpatches.Patch(color="#E53935", label="Attack (1)"),
    ]
    ax.legend(handles=legend_handles, loc="upper right", fontsize=10)
    ax.set_title(f"Flow-Similarity Graph — first {N_SHOW} nodes of graph_00001\n"
                 f"(Edges = k=5 cosine-similarity k-NN on ct_* features)",
                 fontsize=11, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("networkx not installed — skipping graph visualisation.")
    print("Install with:  pip install networkx")

---
## 5. Graph Quality — k Sensitivity Analysis

We built graphs for **k = 3, 5, 10** and recorded structural statistics.  
Results loaded from `data/processed/graph_analysis/quality_analysis.json`.

k = 5 was selected as the primary configuration for all training experiments.

In [ ]:
# ── Load quality analysis results ────────────────────────────────────────────
with open(ANALYSIS_DIR / "quality_analysis.json") as f:
    quality = json.load(f)

rows = []
for k_str in ["3", "5", "10"]:
    for split in ["train", "test"]:
        s   = quality[k_str][split]["structural"]
        sim = quality[k_str][split]["similarity"]
        rows.append({
            "k": int(k_str), "Split": split.capitalize(),
            "Graphs": s["n_graphs"], "Total Edges": s["total_edges"],
            "Avg Degree": s["avg_degree"], "Isolated": s["n_isolated"],
            "Mean Sim": round(sim["mean"], 4), "Median Sim": round(sim["median"], 4),
            "Std Sim": round(sim["std"], 4),
        })

q_df = pd.DataFrame(rows)
display(q_df.style.hide(axis="index")
        .set_caption("Graph quality statistics for k = 3, 5, 10"))

In [ ]:
# ── k vs Mean Similarity and k vs Edge Count ─────────────────────────────────
k_vals = [3, 5, 10]
tr_sim  = [quality[str(k)]["train"]["similarity"]["mean"] for k in k_vals]
te_sim  = [quality[str(k)]["test" ]["similarity"]["mean"] for k in k_vals]
tr_edges = [quality[str(k)]["train"]["structural"]["total_edges"] for k in k_vals]
te_edges = [quality[str(k)]["test" ]["structural"]["total_edges"] for k in k_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("k-Sensitivity Analysis — Graph Quality", fontsize=12, fontweight="bold")

ax1.plot(k_vals, tr_sim, "o-", color="#2c7bb6", linewidth=2, markersize=7, label="Train")
ax1.plot(k_vals, te_sim, "s--", color="#d7191c", linewidth=2, markersize=7, label="Test")
ax1.set_xlabel("k (neighbours per node)"); ax1.set_ylabel("Mean Cosine Similarity")
ax1.set_title("k vs Mean Edge Similarity"); ax1.legend(); ax1.set_xticks(k_vals)
ax1.set_ylim(0.95, 1.0); ax1.grid(alpha=0.3)
ax1.axvline(5, color="grey", linestyle=":", alpha=0.7, label="Selected k=5")

ax2.plot(k_vals, [e/1e6 for e in tr_edges], "o-", color="#2c7bb6", linewidth=2, markersize=7, label="Train")
ax2.plot(k_vals, [e/1e6 for e in te_edges], "s--", color="#d7191c", linewidth=2, markersize=7, label="Test")
ax2.set_xlabel("k (neighbours per node)"); ax2.set_ylabel("Total Edges (millions)")
ax2.set_title("k vs Total Edge Count"); ax2.legend(); ax2.set_xticks(k_vals)
ax2.grid(alpha=0.3)
ax2.axvline(5, color="grey", linestyle=":", alpha=0.7)

plt.tight_layout()
plt.show()

print("\nKey observation: mean similarity is stable across k values.")
print("k=5 was selected as the primary configuration (not claimed as universally optimal).")

---
## 6. GraphSAGE Model Architecture

**GraphSAGE (Graph SAmple and aggreGatE)** is a Graph Neural Network that makes predictions  
by combining a node's own features with an aggregated summary of its neighbours' features.

Each node gathers information from its connected neighbours (message passing),  
so the prediction for a flow depends not just on that flow alone, but on what nearby flows look like.

```
Input  : node feature matrix  (N × 42)
         ↓
SAGEConv layer 1 : 42 → 64   (mean aggregation of 5 neighbours)
         ↓  ReLU + Dropout(0.3)
SAGEConv layer 2 : 64 → 64
         ↓  ReLU + Dropout(0.3)
Linear head      : 64 → 2    (logits for Normal / Attack)
         ↓
Prediction       : argmax → 0 (Normal) or 1 (Attack)
```

In [ ]:
# ── Display model config from saved JSON ─────────────────────────────────────
with open(RESULTS_GNN / "model_config.json") as f:
    cfg = json.load(f)

cfg_df = pd.DataFrame([
    ["Input features",   42,                      "42 preprocessed flow features per node"],
    ["Hidden dimension", cfg["hidden_dim"],         "SAGEConv layer width"],
    ["SAGEConv layers",  cfg["num_layers"],          "Depth of message passing"],
    ["Output classes",   2,                         "Normal (0) / Attack (1)"],
    ["Dropout",          cfg["dropout"],            "Applied after each SAGEConv layer"],
    ["Learning rate",    cfg["lr"],                 "Adam optimizer"],
    ["Weight decay",     cfg["weight_decay"],        "L2 regularisation"],
    ["Epochs",           cfg["epochs"],             "Training epochs"],
    ["Batch size",       cfg["batch_size"],         "Graphs per training step"],
    ["Seed",             cfg["seed"],               "Reproducibility"],
    ["Val split",        cfg["val_split"],          f"{int(cfg['val_split']*100)}% of train graphs held for validation"],
    ["Trainable params", "13,826",                  "Confirmed from model.count_parameters()"],
], columns=["Parameter", "Value", "Notes"])

display(cfg_df.style.hide(axis="index"))

In [ ]:
# ── Instantiate model and show repr ──────────────────────────────────────────
from src.models.graphsage import GraphSAGE
from src.gnn_config import GNN_INPUT_DIM

model = GraphSAGE(
    in_channels = GNN_INPUT_DIM,
    hidden_dim  = cfg["hidden_dim"],
    num_classes = 2,
    num_layers  = cfg["num_layers"],
    dropout     = cfg["dropout"],
)
print(model)

---
## 7. GraphSAGE Training Results

Training history loaded from `results/gnn/training_history.csv`.  
**No retraining is performed here.**

In [ ]:
# ── Load training history ────────────────────────────────────────────────────
hist = pd.read_csv(RESULTS_GNN / "training_history.csv")

best_epoch = hist.loc[hist["val_f1"].idxmax()]
print(f"Total epochs   : {len(hist)}")
print(f"Best val F1    : {best_epoch['val_f1']:.4f}  at epoch {int(best_epoch['epoch'])}")
print(f"Final train F1 : {hist['train_f1'].iloc[-1]:.4f}")
print(f"Final val F1   : {hist['val_f1'].iloc[-1]:.4f}")

In [ ]:
# ── Training curves ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("GraphSAGE Training History — UNSW-NB15", fontsize=13, fontweight="bold")

epochs = hist["epoch"]

# Loss
axes[0].plot(epochs, hist["train_loss"], color="#2c7bb6", linewidth=1.8, label="Train loss")
axes[0].plot(epochs, hist["val_loss"],   color="#d7191c", linewidth=1.8, label="Val loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

# F1
best_ep = int(best_epoch["epoch"])
axes[1].plot(epochs, hist["train_f1"],  color="#2c7bb6", linewidth=1.5, alpha=0.7, label="Train F1")
axes[1].plot(epochs, hist["val_f1"],    color="#1a9641", linewidth=2.2, label="Val F1")
axes[1].plot(epochs, hist["val_precision"], color="#fdae61", linewidth=1.2,
             linestyle="--", label="Val Precision")
axes[1].plot(epochs, hist["val_recall"],    color="#a6611a", linewidth=1.2,
             linestyle=":",  label="Val Recall")
axes[1].axvline(best_ep, color="grey", linestyle="--", alpha=0.6)
axes[1].annotate(f"Best val F1\n{best_epoch['val_f1']:.4f}\n(epoch {best_ep})",
                 xy=(best_ep, best_epoch["val_f1"]),
                 xytext=(best_ep + 2, best_epoch["val_f1"] - 0.04),
                 fontsize=8, color="grey",
                 arrowprops=dict(arrowstyle="->", color="grey", lw=1))
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].set_title("Validation Metrics"); axes[1].legend(fontsize=8)
axes[1].set_ylim(0.5, 1.05); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Also show the saved training history PNG ─────────────────────────────────
png_path = RESULTS_GNN / "training_history.png"
if png_path.exists():
    display(Image(str(png_path), width=800))
else:
    print(f"Not found: {png_path}")

---
## 8. GraphSAGE Test Set Results

Results loaded from `results/gnn/test_metrics.json`.  
The model was evaluated **once** on the official test set after training was complete.

| Metric | Simple meaning |
|---|---|
| **Accuracy** | Fraction of all flows classified correctly |
| **Precision** | Of flows predicted as attacks, how many really are attacks |
| **Recall** | Of all real attacks, how many did we catch (high recall = fewer missed attacks) |
| **F1** | Harmonic mean of precision and recall — balanced overall score |

In [ ]:
# ── Load and display GNN test metrics ────────────────────────────────────────
with open(RESULTS_GNN / "test_metrics.json") as f:
    gnn_m = json.load(f)

print(f"Test set : {gnn_m['total_nodes']:,} flows  "
      f"({gnn_m['n_normal']:,} Normal / {gnn_m['n_attack']:,} Attack)")
print()
metrics_df = pd.DataFrame([
    ["Accuracy",  gnn_m["accuracy"]],
    ["Precision", gnn_m["precision"]],
    ["Recall",    gnn_m["recall"]],
    ["F1-score",  gnn_m["f1"]],
], columns=["Metric", "Score"])
display(metrics_df.style.hide(axis="index").format({"Score": "{:.4f}"}))

# Classification report
report_path = RESULTS_GNN / "classification_report.txt"
if report_path.exists():
    print("\nPer-class report:")
    print(report_path.read_text())

In [ ]:
# ── Confusion matrix ─────────────────────────────────────────────────────────
cm = gnn_m["confusion_matrix"]
cm_arr = np.array(cm)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_arr, cmap="Blues", interpolation="nearest")
plt.colorbar(im, ax=ax)
classes = ["Normal (0)", "Attack (1)"]
ax.set_xticks([0,1]); ax.set_xticklabels(classes, fontsize=10)
ax.set_yticks([0,1]); ax.set_yticklabels(classes, fontsize=10)
ax.set_xlabel("Predicted", fontsize=11); ax.set_ylabel("True", fontsize=11)
ax.set_title("GraphSAGE — Confusion Matrix (Test Set)", fontsize=12, fontweight="bold")
thresh = cm_arr.max() / 2
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm_arr[i,j]:,}", ha="center", va="center",
                color="white" if cm_arr[i,j] > thresh else "black",
                fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Also display saved PNG
cm_png = RESULTS_GNN / "confusion_matrix.png"
if cm_png.exists():
    print("\nSaved confusion matrix image:")
    display(Image(str(cm_png), width=480))

---
## 9. Random Forest Baseline vs GraphSAGE

**Why Random Forest?**  
It provides a conventional ML baseline using the **exact same 42 features** as GraphSAGE,  
but treats every flow as an independent sample — no graph structure, no neighbourhood information.

Comparing the two isolates what, if anything, the graph structure contributes.

Configuration: 200 trees, `class_weight='balanced'`, `random_state=42`.

In [ ]:
# ── Load RF results ───────────────────────────────────────────────────────────
with open(RESULTS_ML / "test_metrics.json") as f:
    rf_m = json.load(f)

comp_df = pd.DataFrame({
    "Metric":        ["Accuracy", "Precision", "Recall", "F1-score"],
    "Random Forest": [rf_m["accuracy"], rf_m["precision"], rf_m["recall"], rf_m["f1"]],
    "GraphSAGE (A)": [gnn_m["accuracy"], gnn_m["precision"], gnn_m["recall"], gnn_m["f1"]],
})
comp_df["Δ (GNN − RF)"] = (comp_df["GraphSAGE (A)"] - comp_df["Random Forest"]).round(4)

display(comp_df.style.hide(axis="index")
        .format({"Random Forest": "{:.4f}", "GraphSAGE (A)": "{:.4f}",
                 "Δ (GNN − RF)": "{:+.4f}"})
        .applymap(lambda v: "color: green" if isinstance(v, float) and v > 0
                  else ("color: red" if isinstance(v, float) and v < 0 else ""),
                  subset=["Δ (GNN − RF)"]))

In [ ]:
# ── Grouped bar chart: RF vs GraphSAGE ───────────────────────────────────────
metrics_labels = ["Accuracy", "Precision", "Recall", "F1"]
rf_vals  = [rf_m["accuracy"],  rf_m["precision"],  rf_m["recall"],  rf_m["f1"]]
gnn_vals = [gnn_m["accuracy"], gnn_m["precision"], gnn_m["recall"], gnn_m["f1"]]

x = np.arange(len(metrics_labels))
width = 0.32

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width/2, rf_vals,  width, label="Random Forest", color="#5C85D6", edgecolor="white")
b2 = ax.bar(x + width/2, gnn_vals, width, label="GraphSAGE (A)", color="#E55353", edgecolor="white")

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(metrics_labels)
ax.set_ylabel("Score"); ax.set_ylim(0.7, 1.05)
ax.set_title("Random Forest vs GraphSAGE — Test Set Comparison\n"
             "(Same 42 features; RF ignores graph structure)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Observation: results are close. GraphSAGE has higher F1 and precision;")
print("Random Forest has higher recall. Neither is the universal winner.")

In [ ]:
# ── Top-15 Random Forest feature importances ─────────────────────────────────
with open(RESULTS_ML / "feature_importances.json") as f:
    imp_dict = json.load(f)

imp_s = pd.Series(imp_dict).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#E53935" if "ct_" in feat else "#1E88E5" for feat in imp_s.index]
ax.barh(imp_s.index[::-1], imp_s.values[::-1], color=colors[::-1], edgecolor="white")
ax.set_xlabel("Feature importance (Gini)")
ax.set_title("Random Forest — Top 15 Feature Importances", fontsize=11, fontweight="bold")
legend_handles = [
    mpatches.Patch(color="#E53935", label="ct_* feature"),
    mpatches.Patch(color="#1E88E5", label="Other feature"),
]
ax.legend(handles=legend_handles, fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Ablation Study

An ablation study **removes or changes one component** to understand its contribution.

We ran **3 experiments** with identical model architecture, features, training procedure, seed and train/val split.  
Only the **graph wiring** changes between experiments.

| Experiment | Graph structure | Purpose |
|---|---|---|
| **A** | Original k=5 cosine-similarity k-NN | Baseline — the real graph |
| **B** | All edges removed | Does any graph structure help at all? |
| **C** | Degree-preserving random rewiring | Does the *specific* similarity wiring matter? |

**What stays identical:** architecture, 42 features, loss, optimizer, epochs, seed, train/val split.

In [ ]:
# ── Load ablation results ────────────────────────────────────────────────────
with open(ABLATION_DIR / "A_original_metrics.json") as f: a_m = json.load(f)
with open(ABLATION_DIR / "B_no_edges_metrics.json") as f: b_m = json.load(f)
with open(ABLATION_DIR / "C_random_metrics.json")   as f: c_m = json.load(f)
with open(ABLATION_DIR / "comparison_table.json")   as f: comp = json.load(f)

abl_df = pd.DataFrame([
    {
        "Experiment": r["experiment"],
        "Graph Structure": r["graph_structure"],
        "Accuracy":  r["accuracy"],
        "Precision": r["precision"],
        "Recall":    r["recall"],
        "F1":        r["f1"],
        "Best Val F1": r["best_val_f1"],
    }
    for r in comp
])
display(abl_df.style.hide(axis="index")
        .format({"Accuracy":"{:.4f}","Precision":"{:.4f}",
                 "Recall":"{:.4f}","F1":"{:.4f}","Best Val F1":"{:.4f}"}))

In [ ]:
# ── Grouped bar chart A / B / C ───────────────────────────────────────────────
exp_labels   = ["A — Original k-NN", "B — No edges", "C — Random rewiring"]
exp_colors   = ["#1E88E5", "#FB8C00", "#43A047"]
metric_keys  = ["Accuracy", "Precision", "Recall", "F1"]
metric_vals  = [
    [a_m["accuracy"],  b_m["accuracy"],  c_m["accuracy"]],
    [a_m["precision"], b_m["precision"], c_m["precision"]],
    [a_m["recall"],    b_m["recall"],    c_m["recall"]],
    [a_m["f1"],        b_m["f1"],        c_m["f1"]],
]

x     = np.arange(len(metric_keys))
width = 0.24

fig, ax = plt.subplots(figsize=(11, 5))
for i, (label, color, vals) in enumerate(zip(exp_labels, exp_colors, zip(*metric_vals))):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, vals, width, label=label, color=color, edgecolor="white")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x); ax.set_xticklabels(metric_keys)
ax.set_ylabel("Score"); ax.set_ylim(0.75, 1.02)
ax.set_title("Ablation Study — A vs B vs C\n"
             "(Identical architecture/features/training; only graph wiring differs)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices for A, B, C ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Ablation Study — Confusion Matrices (Test Set)",
             fontsize=12, fontweight="bold")

titles = ["A — Original k-NN", "B — No edges", "C — Random rewiring"]
cms    = [a_m["confusion_matrix"], b_m["confusion_matrix"], c_m["confusion_matrix"]]
classes = ["Normal", "Attack"]

for ax, cm_data, title in zip(axes, cms, titles):
    arr = np.array(cm_data)
    im = ax.imshow(arr, cmap="Blues", interpolation="nearest")
    ax.set_xticks([0,1]); ax.set_xticklabels(classes, fontsize=9)
    ax.set_yticks([0,1]); ax.set_yticklabels(classes, fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title, fontsize=10, fontweight="bold")
    thresh = arr.max() / 2
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{arr[i,j]:,}", ha="center", va="center",
                    color="white" if arr[i,j] > thresh else "black",
                    fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

---
## 11. Ablation Graph Validation

Before running Experiment C, we verified that the degree-preserving rewiring was correct.  
The following properties were confirmed on 50 training graphs:

| Property | Result |
|---|---|
| Edge count preserved (C == A) | ✅ 25,000 = 25,000 |
| Self-loops | ✅ 0 |
| Duplicate edges | ✅ 0 |
| Out-degree mismatches | ✅ 0 |
| Node features (x) modified | ✅ No |
| Labels (y) modified | ✅ No |
| attack_cat modified | ✅ No |
| Rewiring independent of labels | ✅ Confirmed |

**24 unit tests** in `tests/test_ablation.py` — all passing — cover these structural guarantees.

In [ ]:
# ── Display ablation training histories (val F1 over epochs) ─────────────────
fig, ax = plt.subplots(figsize=(11, 4))

colors_abc = {"A": "#1E88E5", "B": "#FB8C00", "C": "#43A047"}
labels_abc = {
    "A": f"A — Original k-NN (best val F1={comp[0]['best_val_f1']:.4f})",
    "B": f"B — No edges       (best val F1={comp[1]['best_val_f1']:.4f})",
    "C": f"C — Random rewiring(best val F1={comp[2]['best_val_f1']:.4f})",
}

for key, fname in [("A", "A_original_history.json"),
                   ("B", "B_no_edges_history.json"),
                   ("C", "C_random_history.json")]:
    with open(ABLATION_DIR / fname) as f:
        h = json.load(f)
    ep = [r["epoch"] for r in h]
    vf = [r["val_f1"] for r in h]
    ax.plot(ep, vf, linewidth=2, color=colors_abc[key], label=labels_abc[key])

ax.set_xlabel("Epoch"); ax.set_ylabel("Validation F1")
ax.set_title("Ablation Study — Validation F1 per Epoch", fontsize=12, fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

---
## 12. Why Did A, B, and C Perform Differently?

The following measurements were made directly from the actual graph data  
(50 training graphs, 5,000 nodes, 25,000 edges each).  
These are **observed measurements** — not proven causal laws.

### A > B: Graph structure helps
Removing all edges (B) drops F1 from 0.8991 → 0.8688.  
SAGEConv without neighbours degenerates to a per-node linear transformation.

### C > A: The *specific* similarity wiring matters differently than expected
The experiments observed that the k-NN graph (A) intentionally connects nodes that are  
**nearly identical** in the ct_* dimensions, leaving very little new information in the aggregated neighbourhood.  
The degree-preserving random wiring (C) creates a **more diverse neighbourhood** within the same window.

In [ ]:
# ── Structural analysis table (from ablation_report.md measurements) ──────────
analysis_df = pd.DataFrame([
    ["Mean abs ct_* diff: node vs neighbour mean",
     "0.0613", "—", "0.3085", "5.0×",
     "kNN neighbours are near-identical in ct_* space → neighbourhood adds little new info"],
    ["Mean abs ct_* diff (mixed-label windows)",
     "0.0427", "—", "0.3403", "8.0×",
     "Effect amplified in windows with both Normal and Attack nodes"],
    ["Cross-class connectivity (mixed-label windows)",
     "6.9%", "—", "29.8%", "4.3×",
     "C connects across class boundary 4× more often than A"],
    ["L2 distance: node vs neighbourhood mean (42-dim)",
     "5.035", "—", "6.469", "1.28×",
     "C neighbourhood provides 28% more independent information overall"],
    ["Overall neighbour feature variance (42-dim)",
     "303.96", "—", "303.98", "1.00×",
     "Total feature diversity is EQUAL — C redistributes, does not inject extra info"],
], columns=["Measurement", "Exp A", "Exp B", "Exp C", "C/A Ratio", "Interpretation"])

display(analysis_df.style.hide(axis="index")
        .set_properties(**{"text-align": "left", "max-width": "300px"}))

In [ ]:
# ── Visual summary of the key structural differences ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Structural Differences: Exp A vs Exp C", fontsize=12, fontweight="bold")

# ct_* contrast
axes[0].bar(["A — k-NN", "C — Random"], [0.0613, 0.3085],
            color=["#1E88E5", "#43A047"], width=0.4, edgecolor="white")
axes[0].set_title("Mean |ct_* diff|: Node vs Neighbourhood Mean", fontsize=10)
axes[0].set_ylabel("Mean absolute difference")
axes[0].set_ylim(0, 0.38)
for i, v in enumerate([0.0613, 0.3085]):
    axes[0].text(i, v + 0.008, f"{v}", ha="center", fontsize=11, fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)
axes[0].annotate("5.0× more contrast in C", xy=(1, 0.3085), xytext=(0.5, 0.34),
                 fontsize=9, color="#43A047", ha="center")

# Cross-class connectivity
axes[1].bar(["A — k-NN", "C — Random"], [6.88, 29.79],
            color=["#1E88E5", "#43A047"], width=0.4, edgecolor="white")
axes[1].set_title("Cross-Class Connectivity\n(% neighbours with different label)", fontsize=10)
axes[1].set_ylabel("% of edges crossing class boundary")
axes[1].set_ylim(0, 38)
for i, v in enumerate([6.88, 29.79]):
    axes[1].text(i, v + 0.8, f"{v:.1f}%", ha="center", fontsize=11, fontweight="bold")
axes[1].grid(axis="y", alpha=0.3)
axes[1].annotate("4.3× more cross-class\nconnections in C", xy=(1, 29.79), xytext=(0.5, 33),
                 fontsize=9, color="#43A047", ha="center")

plt.tight_layout()
plt.show()

print("Summary: The k-NN wiring connects near-identical ct_* flows, so the neighbourhood")
print("mean adds very little new information. The random rewiring exposes nodes to a more")
print("diverse neighbourhood — the same total feature diversity, just redistributed.")
print("\nNOTE: These are observed measurements, not proven causal laws.")

---
## 13. Inference Pipeline

The inference pipeline (`src/inference.py`) allows **any CSV file** to be processed  
and classified using the trained GraphSAGE model — without retraining or re-fitting.

```
New CSV file
    ↓  load_inference_csv()   — validates schema; injects dummy targets if absent
    ↓  build_inference_graphs() — applies saved preprocessor.pkl; builds k-NN graphs
    ↓  load_graphsage()       — loads models/graphsage_best.pt; eval() mode
    ↓  run_inference()        — torch.no_grad(); softmax probabilities per flow
    ↓  summarise()            — total / normal / attack counts and percentages
    →  predict_csv()          — end-to-end function: CSV path → predictions dict
```

This is the **bridge to the future web application**: the backend will call `predict_csv()`  
when a user uploads a CSV file.

In [ ]:
# ── Demonstrate inference pipeline on a small slice of the training CSV ───────
# We use 250 rows from the TRAINING CSV as a stand-in for a "new uploaded CSV"
# (the inference pipeline treats any CSV the same way)

import tempfile
from src.inference import predict_csv
from src.config import TRAIN_RAW_PATH

# Write a 250-row sample to a temp file
sample_df = pd.read_csv(TRAIN_RAW_PATH, nrows=250)
with tempfile.NamedTemporaryFile(suffix=".csv", delete=False, mode="w") as tmp:
    sample_df.to_csv(tmp, index=False)
    tmp_path = tmp.name

print(f"Running inference on 250-row sample CSV: {tmp_path}")
result = predict_csv(tmp_path)

s = result["summary"]
print(f"\n── Inference Summary ───────────────────")
print(f"  Total flows    : {s['total_flows']:>6,}")
print(f"  Normal flows   : {s['normal_flows']:>6,}")
print(f"  Attack flows   : {s['attack_flows']:>6,}")
print(f"  Attack %       : {s['attack_pct']:>6.1f}%")
print(f"  Graphs created : {result['n_graphs']:>6}")

In [ ]:
# ── Show first 10 per-flow predictions with probabilities ─────────────────────
preds = result["predictions"]
probs = result["probabilities"]

pred_df = pd.DataFrame({
    "flow_index":  range(10),
    "prediction":  ["Normal" if p == 0 else "Attack" for p in preds[:10]],
    "P(Normal)":   probs[:10, 0].round(4),
    "P(Attack)":   probs[:10, 1].round(4),
})
display(pred_df.style.hide(axis="index")
        .applymap(lambda v: "background-color: #ffcccc" if v == "Attack" else
                           ("background-color: #ccffcc" if v == "Normal" else ""),
                  subset=["prediction"])
        .set_caption("First 10 flow predictions from inference pipeline"))

# Clean up temp file
import os; os.unlink(tmp_path)
print("\nInference pipeline works correctly on a new CSV.")

---
## 14. Automated Tests

The project has a comprehensive pytest test suite covering every module.  
Test counts were verified directly from the repository.

In [ ]:
# ── Display test summary table ────────────────────────────────────────────────
test_summary = pd.DataFrame([
    ["test_data_loader.py",    17, "CSV loading, column validation, error handling"],
    ["test_preprocessor.py",   17, "StandardScaler, OrdinalEncoder, pipeline persistence"],
    ["test_graph_builder.py",  25, "Node/edge count, feature dims, leakage, save/load"],
    ["test_graph_validator.py",21, "Graph stats, leakage detection, file output"],
    ["test_graphsage_model.py",25, "Output shape, binary preds, class weights, metrics"],
    ["test_random_forest.py",  22, "42-feature integrity, training, eval, persistence"],
    ["test_ablation.py",       24, "Edge count, degree seq, no self-loops, label independence"],
    ["test_inference.py",      33, "End-to-end CSV→prediction, schema handling, eval mode"],
], columns=["Test File", "Tests", "Coverage"])

total = test_summary["Tests"].sum()
display(test_summary.style.hide(axis="index")
        .set_caption(f"Total: {total} automated tests across 8 modules"))
print(f"\nTotal automated tests: {total}")

---
## 15. Current Project Status

### ✅ Completed

In [ ]:
completed = pd.DataFrame([
    ["✅", "Dataset integration",       "UNSW-NB15 training (175,341) + testing (82,332) CSVs"],
    ["✅", "Data inspection",           "Shape, dtypes, missing values, target distributions, leakage check"],
    ["✅", "Preprocessing pipeline",    "StandardScaler + OrdinalEncoder → 42 features, saved to preprocessor.pkl"],
    ["✅", "Graph construction",        "1,754 train + 824 test PyG graphs (window=100, k=5)"],
    ["✅", "Graph validation",          "Leakage checks, structural stats, per-graph CSV/JSON reports"],
    ["✅", "Graph quality analysis",    "k = 3, 5, 10 sensitivity study with plots"],
    ["✅", "GraphSAGE model",           "SAGEConv×2 + Linear head, 13,826 params"],
    ["✅", "GNN training",              "50 epochs, Adam, weighted CrossEntropyLoss, best-checkpoint saving"],
    ["✅", "GNN evaluation",            "F1=0.8994, Acc=0.8817 on official test set"],
    ["✅", "Random Forest baseline",    "200 trees, same 42 features, F1=0.8941"],
    ["✅", "Ablation study",            "3 experiments (A/B/C), comparison table, confusion matrices"],
    ["✅", "Ablation validation",       "Degree-preserving rewiring verified; 24 tests passing"],
    ["✅", "Inference pipeline",        "predict_csv() — any CSV → Normal/Attack per flow + probabilities"],
    ["✅", "Automated test suite",      "184 tests across 8 modules"],
    ["✅", "Documentation",            "README.md, PROJECT_STATUS.md"],
], columns=["Status", "Component", "Details"])

display(completed.style.hide(axis="index"))

In [ ]:
remaining = pd.DataFrame([
    ["⏳", "Web frontend",           "React (or similar) — CSV upload UI, results dashboard"],
    ["⏳", "Backend API",            "Flask/FastAPI endpoint calling predict_csv()"],
    ["⏳", "Result dashboard",       "Attack counts, percentages, per-flow probability table"],
    ["⏳", "Graph visualisation",    "Interactive display of a sample graph from uploaded CSV"],
    ["⏳", "System integration",     "End-to-end browser → prediction → display test"],
    ["⏳", "Deployment",             "Server/cloud setup for final demo"],
    ["⏳", "Final project report",   "Academic write-up: methodology, results, discussion"],
], columns=["Status", "Component", "Details"])

print("Not yet implemented:")
display(remaining.style.hide(axis="index"))

---
## 16. Planned Web Platform  *(Future Work)*

> ⚠️ The web platform is **not yet implemented**. This section describes the planned architecture.

```
  USER (browser)
       ↓
  CSV UPLOAD  ←──────────── Any network-flow CSV file
       ↓
  BACKEND (Flask / FastAPI)
       ↓
  PREPROCESSING  ←────────── preprocessor.pkl (already built)
       ↓
  GRAPH CONSTRUCTION  ←───── graph_builder.py (already built)
       ↓
  GRAPHSAGE INFERENCE  ←──── graphsage_best.pt (already trained)
       ↓
  predict_csv() returns predictions + probabilities
       ↓
  RESULTS DASHBOARD
       ├── Total flows / Normal / Attack counts
       ├── Attack percentage
       ├── Per-flow prediction table with probabilities
       ├── Attack category breakdown (if available)
       └── Graph visualisation (sample window)
```

The entire ML backend is complete. The remaining work is building the web layer on top of it.

---
## 17. Complete Results Summary

All metrics loaded directly from saved result files — no retraining.

In [ ]:
# ── Master results table ──────────────────────────────────────────────────────
summary_df = pd.DataFrame([
    ["Random Forest",    "No graph",                   rf_m["accuracy"], rf_m["precision"], rf_m["recall"], rf_m["f1"]],
    ["GraphSAGE (A)",    "k=5 cosine-sim k-NN",        a_m["accuracy"], a_m["precision"], a_m["recall"], a_m["f1"]],
    ["GraphSAGE (B)",    "No edges",                   b_m["accuracy"], b_m["precision"], b_m["recall"], b_m["f1"]],
    ["GraphSAGE (C)",    "Degree-preserving random",   c_m["accuracy"], c_m["precision"], c_m["recall"], c_m["f1"]],
], columns=["Model", "Graph Structure", "Accuracy", "Precision", "Recall", "F1"])

display(summary_df.style.hide(axis="index")
        .format({"Accuracy":"{:.4f}","Precision":"{:.4f}","Recall":"{:.4f}","F1":"{:.4f}"})
        .highlight_max(subset=["Accuracy","Precision","Recall","F1"], color="#c8e6c9")
        .set_caption("All experiments — Test set (82,332 flows)"))

print("\nTest set: 82,332 flows  |  Normal: 37,000  |  Attack: 45,332")
print("\nKey finding: Degree-preserving random rewiring (C) outperforms the cosine-")
print("similarity k-NN graph (A). Measured reason: k-NN connects near-identical ct_*")
print("neighbours, leaving minimal new information in the aggregated neighbourhood.")
print("This is an observed experimental result, not a universal claim.")

In [ ]:
# ── Final comparison plot: all 4 models ──────────────────────────────────────
model_names = ["Random\nForest", "GraphSAGE\n(A) k-NN", "GraphSAGE\n(B) No edges", "GraphSAGE\n(C) Random"]
model_colors = ["#5C85D6", "#1E88E5", "#FB8C00", "#43A047"]
f1_scores = [rf_m["f1"], a_m["f1"], b_m["f1"], c_m["f1"]]
acc_scores = [rf_m["accuracy"], a_m["accuracy"], b_m["accuracy"], c_m["accuracy"]]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - width/2, acc_scores, width, label="Accuracy", color=model_colors, alpha=0.6, edgecolor="white")
b2 = ax.bar(x + width/2, f1_scores,  width, label="F1-score", color=model_colors, alpha=1.0, edgecolor="white")

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel("Score"); ax.set_ylim(0.78, 1.02)
ax.set_title("All Models — Accuracy and F1 on Test Set (82,332 flows)",
             fontsize=12, fontweight="bold")
from matplotlib.lines import Line2D
legend_handles = [
    mpatches.Patch(color="grey", alpha=0.6, label="Accuracy (faded)"),
    mpatches.Patch(color="grey", alpha=1.0, label="F1-score (solid)"),
]
ax.legend(handles=legend_handles, fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---
## 18. Summary

### What we have actually built

```
UNSW-NB15 CSV (257,673 labelled flows)
    ↓
Inspection  →  0 missing values, 0 duplicates, 10 attack categories confirmed
    ↓
Preprocessing  →  42 features (StandardScaler + OrdinalEncoder)  →  preprocessor.pkl
    ↓
Graph construction  →  1,754 train + 824 test flow-similarity graphs (window=100, k=5)
    ↓
GraphSAGE training  →  SAGEConv×2, 13,826 params, 50 epochs  →  graphsage_best.pt
    ↓
Evaluation  →  F1=0.8994, Acc=0.8817 on 82,332 test flows
    ↓
Random Forest baseline  →  F1=0.8941 (same 42 features, no graph)
    ↓
Ablation study  →  3 experiments verified; C (random rewiring) F1=0.9546
    ↓
Inference pipeline  →  predict_csv() — any CSV → per-flow predictions
    ↓
Test suite  →  184 automated tests, all passing
```

### What we are building next

```
Web platform
    ↓
CSV upload interface
    ↓
Backend API  →  calls predict_csv()  (bridge already built)
    ↓
Result dashboard  →  attack counts, probabilities, breakdown
    ↓
Graph visualisation  →  interactive sample window graph
    ↓
Integration testing  +  deployment  +  final report
```

---
*All metrics in this notebook are loaded from actual saved result files.  
No models were retrained during notebook execution.*